# Preprocesado de los 1000 casos (Dataset503_ImageCAS1000)

El comando `nnUNetv2_plan_and_preprocess` es una tarea de CPU (fingerprinting y remuestreo), por lo que este notebook no requiere GPU y puede ejecutarse en un entorno sin acelerador, preservando la cuota de GPU para los entrenamientos.

**Estrategia de almacenamiento** (distinta de la empleada con los 200 casos):
- El **raw** (imágenes y máscaras originales, ≈ 83 GB) se descomprime en el **disco local de Colab** (`/content/`) y no en Drive, ya que descomprimir miles de archivos directamente sobre Drive es muy lento (se gestiona archivo por archivo mediante la API).
- El **preprocesado** (datos necesarios para el entrenamiento posterior) se escribe directamente en **Drive**, para que persista entre sesiones.
- Una desconexión de la sesión no supone pérdida de información crítica: el raw puede volver a descomprimirse en un solo paso desde el ZIP, conservado en Drive.


In [ ]:
!df -h /content
# Comprobación de al menos ≈ 90-100 GB libres en el disco local antes de descomprimir

Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   21G  206G  10% /


In [ ]:
!python -m pip install -q nnunetv2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.5 MB/s eta 0:00:00


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

# RAW en disco local (rápido, efímero, regenerable desde el ZIP en Drive)
NNUNET_RAW = Path("/content/nnUNet_raw")
NNUNET_RAW.mkdir(parents=True, exist_ok=True)

# PREPROCESSED en Drive (persistente; es el dato que debe conservarse)
NNUNET_PREPROCESSED = BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE / "nnUNet_results"

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)

DATASET_ID = 503
DATASET_FOLDER_NAME = "Dataset503_ImageCAS1000"
ZIP_PATH = BASE / "Dataset503_ImageCAS1000.zip"

print("nnUNet_raw (local):", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed (Drive):", os.environ["nnUNet_preprocessed"])
print("Zip existe:", ZIP_PATH.exists(), "-", round(ZIP_PATH.stat().st_size / 1e9, 2), "GB" if ZIP_PATH.exists() else "")

nnUNet_raw (local): /content/nnUNet_raw
nnUNet_preprocessed (Drive): /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed
Zip existe: True - 89.16 GB


## 1. Descompresión del raw en el disco local

Se utiliza `unzip` del sistema, mucho más rápido que `zipfile` de Python para volúmenes grandes. La opción `-q` suprime el listado de archivos (miles de líneas con 1000 casos) y puede omitirse para seguir el progreso caso a caso.


In [ ]:
import time

inicio = time.time()
!unzip -q "{ZIP_PATH}" -d "{NNUNET_RAW}"
print(f"Descomprimido en {(time.time() - inicio) / 60:.1f} min")

Descomprimido en 44.1 min


## 2. Verificación de la estructura extraída

Se comprueba que el ZIP se descomprimió con la estructura que espera nnU-Net: una carpeta `Dataset503_ImageCAS1000/` con `imagesTr/`, `labelsTr/` y `dataset.json` directamente en su interior, sin un nivel adicional de anidación (error frecuente en ZIP exportados desde distintas herramientas).


In [ ]:
print("Contenido de", NNUNET_RAW, ":")
for item in sorted(NNUNET_RAW.iterdir()):
    print(" -", item.name)

ruta_dataset = NNUNET_RAW / DATASET_FOLDER_NAME
print(f"\n¿Existe {DATASET_FOLDER_NAME}/? -> {ruta_dataset.exists()}")

if ruta_dataset.exists():
    print("\nContenido de", ruta_dataset, ":")
    for item in sorted(ruta_dataset.iterdir()):
        print(" -", item.name)

Contenido de /content/nnUNet_raw :
 - Dataset503_ImageCAS1000

¿Existe Dataset503_ImageCAS1000/? -> True

Contenido de /content/nnUNet_raw/Dataset503_ImageCAS1000 :
 - dataset.json
 - imagesTr
 - labelsTr


## 3. Corrección de estructura anidada (si procede)

Si la celda anterior muestra una estructura anidada, como `Dataset503_ImageCAS1000/Dataset503_ImageCAS1000/imagesTr/...` (doble carpeta) o `Dataset503_ImageCAS1000/ImageCAS1000_raw/imagesTr/...`, se corrige aquí ajustando `SUBCARPETA_REAL` al nombre exacto de la subcarpeta detectada:


In [ ]:
# Solo se ejecuta si la estructura resultó anidada; si imagesTr, labelsTr y dataset.json
# están directamente dentro de Dataset503_ImageCAS1000/, esta celda no es necesaria

# SUBCARPETA_REAL = "nombre_de_la_subcarpeta_anidada_detectada_arriba"
# import shutil
# origen = ruta_dataset / SUBCARPETA_REAL
# for item in origen.iterdir():
#     shutil.move(str(item), str(ruta_dataset / item.name))
# origen.rmdir()
# print("Estructura corregida.")

In [ ]:
# Conteo de casos: se espera ≈ 1000 y coincidencia entre imágenes y máscaras
imagenes = sorted((ruta_dataset / "imagesTr").glob("*.nii.gz"))
mascaras = sorted((ruta_dataset / "labelsTr").glob("*.nii.gz"))

print("Imagenes en imagesTr:", len(imagenes))
print("Mascaras en labelsTr:", len(mascaras))

nombres_img = {f.name.replace("_0000.nii.gz", "") for f in imagenes}
nombres_seg = {f.name.replace(".nii.gz", "") for f in mascaras}

faltan_seg = sorted(nombres_img - nombres_seg)
faltan_img = sorted(nombres_seg - nombres_img)

print("Casos con imagen pero sin mascara:", faltan_seg[:10], "..." if len(faltan_seg) > 10 else "")
print("Casos con mascara pero sin imagen:", faltan_img[:10], "..." if len(faltan_img) > 10 else "")

print("\ndataset.json existe:", (ruta_dataset / "dataset.json").exists())
if (ruta_dataset / "dataset.json").exists():
    import json as _json
    dj = _json.loads((ruta_dataset / "dataset.json").read_text())
    print("numTraining declarado en dataset.json:", dj.get("numTraining"))

Imagenes en imagesTr: 1000
Mascaras en labelsTr: 1000
Casos con imagen pero sin mascara: [] 
Casos con mascara pero sin imagen: [] 

dataset.json existe: True
numTraining declarado en dataset.json: 1000


## 4. Preprocesado

Los parámetros `-npfp` (procesos de fingerprinting) y `-np` (procesos de preprocesado propiamente dicho) pueden aumentarse respecto a los valores 2/2 empleados con 200 casos si la instancia de Colab dispone de más CPUs (se consulta con `!nproc`). Con 1000 casos (5 veces más) se espera un tiempo claramente superior; al no existir una referencia previa para este tamaño, se cronometra la ejecución.


In [ ]:
!nproc

8


In [8]:
import time

inicio = time.time()
!nnUNetv2_plan_and_preprocess \
-d 503 \
-c 3d_fullres \
--verify_dataset_integrity \
-npfp 4 \
-np 4

print(f"\nPreprocesado completo en {(time.time() - inicio) / 60:.1f} min")

Fingerprint extraction...
Dataset503_ImageCAS1000
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Extracting dataset fingerprint: 100% 1000/1000 [19:31<00:00,  1.17s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [0.515      0.36009766 0.36009766]. 
Current patch size: (np.int64(96), np.int64(160), np.int64(160)). 
Current median shape: [266.99029126 497.08737864 497.08737864]
Attempting to find 3d_lowres 

## 5. Verificación final del preprocesado (en Drive)


In [9]:
ruta_3d = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / "nnUNetPlans_3d_fullres"

print("Existe 3D FullRes:", ruta_3d.exists())

if ruta_3d.exists():
    imagenes_3d = [f for f in ruta_3d.glob("*.b2nd") if not f.name.endswith("_seg.b2nd")]
    mascaras_3d = list(ruta_3d.glob("*_seg.b2nd"))
    print("Imagenes preprocesadas:", len(imagenes_3d))
    print("Mascaras preprocesadas:", len(mascaras_3d))

gt_folder = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / "gt_segmentations"
print("\ngt_segmentations existe:", gt_folder.exists())
if gt_folder.exists():
    print("Casos en gt_segmentations:", len(list(gt_folder.glob("*.nii.gz"))))

Existe 3D FullRes: True
Imagenes preprocesadas: 999
Mascaras preprocesadas: 1000

gt_segmentations existe: True
Casos en gt_segmentations: 1000


In [11]:
imagenes_3d_nombres = {f.name.replace(".b2nd", "") for f in imagenes_3d}
mascaras_3d_nombres = {f.name.replace("_seg.b2nd", "") for f in mascaras_3d}

falta_imagen = mascaras_3d_nombres - imagenes_3d_nombres
sobra_imagen = imagenes_3d_nombres - mascaras_3d_nombres

print("Caso(s) con mascara pero SIN imagen:", falta_imagen)
print("Caso(s) con imagen pero SIN mascara:", sobra_imagen)
print("Total imagenes:", len(imagenes_3d_nombres), "| Total mascaras:", len(mascaras_3d_nombres))

Caso(s) con mascara pero SIN imagen: {'ImageCAS_0256'}
Caso(s) con imagen pero SIN mascara: set()
Total imagenes: 999 | Total mascaras: 1000


In [12]:
print(list(NNUNET_RAW.glob(f"{DATASET_FOLDER_NAME}/imagesTr/ImageCAS_0256*")))
print(list(NNUNET_RAW.glob(f"{DATASET_FOLDER_NAME}/labelsTr/ImageCAS_0256*")))

[PosixPath('/content/nnUNet_raw/Dataset503_ImageCAS1000/imagesTr/ImageCAS_0256_0000.nii.gz')]
[PosixPath('/content/nnUNet_raw/Dataset503_ImageCAS1000/labelsTr/ImageCAS_0256.nii.gz')]


In [13]:
# Eliminación de la máscara huérfana de ImageCAS_0256 en el preprocesado (su imagen no se generó)
ruta_3d = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / "nnUNetPlans_3d_fullres"
mascara_huerfana = ruta_3d / "ImageCAS_0256_seg.b2nd"

if mascara_huerfana.exists():
    mascara_huerfana.unlink()
    print("Mascara huerfana borrada:", mascara_huerfana)

# También se elimina de gt_segmentations (se usa para evaluación, no para entrenamiento)
gt_huerfano = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / "gt_segmentations" / "ImageCAS_0256.nii.gz"
if gt_huerfano.exists():
    gt_huerfano.unlink()
    print("GT huerfano borrado:", gt_huerfano)

# Verificación final
imagenes_3d = [f for f in ruta_3d.glob("*.b2nd") if not f.name.endswith("_seg.b2nd")]
mascaras_3d = list(ruta_3d.glob("*_seg.b2nd"))
print(f"\nTotal final -> Imagenes: {len(imagenes_3d)}  Mascaras: {len(mascaras_3d)}")

Mascara huerfana borrada: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000/nnUNetPlans_3d_fullres/ImageCAS_0256_seg.b2nd
GT huerfano borrado: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000/gt_segmentations/ImageCAS_0256.nii.gz

Total final -> Imagenes: 999  Mascaras: 999


## 6. Siguiente paso

Con el preprocesado completo y verificado en Drive, se entrena la configuración v1 (λ = 0.2, pesos 1/1) sobre `Dataset503_ImageCAS1000`, igual que con el conjunto de 200 casos: solo cambian `DATASET_ID` y `DATASET_FOLDER_NAME` (de 501 a 503) en el trainer y en el comando de `nnUNetv2_train`.

Al abrir una nueva sesión de Colab para entrenar, `nnUNet_raw` (disco local) no persiste, pero tampoco es necesario: el entrenamiento lee de `nnUNet_preprocessed`, almacenado en Drive. El raw solo debe descomprimirse de nuevo si se ejecuta `plan_and_preprocess` desde cero.
